In [ ]:
pip install requests==2.32.4

In [ ]:
!pip install langchain langchain-community langchain-huggingface huggingface_hub duckduckgo-search requests

In [ ]:
!pip install -U langchain langchain-huggingface huggingface_hub ddgs requests

In [ ]:
import os
import json
import requests
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "XXXXX"
@tool
def get_weather_data(city: str) -> str:
    """Get current weather for a city"""

    url = f"https://api.weatherstack.com/current?access_key=xxxxxxxx&query={city}"

    data = requests.get(url).json()

    if "current" not in data:
        return f"Could not fetch weather for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

# ==================================
# MODEL
# ==================================

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=512,
    temperature=0.1,
)

chat_model = ChatHuggingFace(llm=llm)

# ==================================
# STEP 1
# Ask model to identify city
# ==================================

question = "Find the capital of Andhra Pradesh and tell me its current weather."

response = chat_model.invoke(
    f"""
    Answer the following question.

    Question:
    {question}

    First identify the capital city.
    Return only the city name.
    """
)

city = response.content.strip()
print("Capital:", city)
weather_info = get_weather_data.invoke(city)

print("\nWeather Data:")
print(weather_info)
final_response = chat_model.invoke(
    f"""
    User Question:
    {question}

    Capital City:
    {city}

    Weather Information:
    {weather_info}

    Generate a clear final answer.
    """
)

print("\nFinal Answer:")
print(final_response.content)

Capital: Hyderabad

Weather Data:
City: Hyderabad
Temperature: 30°C
Weather: Haze
Humidity: 62%

Final Answer:
The capital of Andhra Pradesh is Hyderabad. Currently, the weather in Hyderabad is 30°C with haze, and the humidity is 62%.
